# KAN as feature selector

In [9]:
import os
import torch
import numpy as np
import pandas as pd
from kan import KAN, ex_round
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

fold_path = "/home/alecacciatore/ECML26/GNN4eGFR"
out_path = os.path.join(fold_path, "features_importance_score/kan_scores")
file_path = os.path.join(fold_path, "XY_temp.csv")

if not os.path.exists(out_path):
    os.makedirs(out_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Get data

In [10]:
# read the CSV file into a DataFrame
df = pd.read_csv(file_path)

# the last column is the target variable and the first one is an indppex
X = df.iloc[:, 1:-3] # TODO:: include general practitioner features?
y_classif = df.iloc[:, -1]
y_regress = df.iloc[:, -2]

# y_classif contains [I, II, IIIa, IIIb, IV, V] labels
# convert them to numerical labels for classification
# label_mapping = {'I': 0, 'II': 1, 'IIIa': 2, 'IIIb': 3, 'IV': 4, 'V': 5}
label_mapping = {'I': 0, 'II': 1, 'IIIa': 1, 'IIIb': 1, 'IV': 1, 'V': 1}
y_classif = y_classif.map(label_mapping)

# Split data into training and testing sets
X_train, X_test, y_classif_train, y_classif_test = train_test_split(
    X, y_classif, test_size=0.2, random_state=42, stratify=y_classif
)
X_train_reg, X_test_reg, y_regress_train, y_regress_test = train_test_split(
    X, y_regress, test_size=0.2, random_state=42
)

# Create a dataset dictionary for KAN ('test_input', 'test_label', 'train_input', 'train_label')
train_data_classif = {
    'train_input': torch.tensor(X_train.values, dtype=torch.float32).to(device),
    'train_label': torch.tensor(y_classif_train.values, dtype=torch.long).to(device),
    'test_input': torch.tensor(X_test.values, dtype=torch.float32).to(device),
    'test_label': torch.tensor(y_classif_test.values, dtype=torch.long).to(device)
}
train_data_regress = {
    'train_input': torch.tensor(X_train_reg.values, dtype=torch.float32).to(device),
    'train_label': torch.tensor(y_regress_train.values, dtype=torch.float32).to(device),
    'test_input': torch.tensor(X_test_reg.values, dtype=torch.float32).to(device),
    'test_label': torch.tensor(y_regress_test.values, dtype=torch.float32).to(device)
}

In [ ]:
# print per-class distribution and imbalance ratio
class_counts = y_classif_train.value_counts().sort_index()
imbalance_ratio = class_counts.max() / class_counts.min()
print("Class distribution in training set:")
for cls, count in class_counts.items():
    print(f"Class {cls}: {count} samples")
print(f"Imbalance Ratio: {imbalance_ratio:.2f}")

# Compute imbalance ratio
value_counts = y_classif.value_counts()
imbalance_ratio = value_counts.max() / value_counts.min()
print(f"Imbalance ratio: {imbalance_ratio}")

# smote = SMOTE(random_state=42)
# X, y_classif = smote.fit_resample(X, y_classif)

# # New imbalance ratio
# value_counts_resampled = y_classif.value_counts()
# imbalance_ratio_resampled = value_counts_resampled.max() / value_counts_resampled.min()
# print(f"Imbalance ratio after SMOTE: {imbalance_ratio_resampled}")


Class distribution in training set:
Class 0: 1320 samples
Class 1: 6012 samples
Imbalance Ratio: 4.55
Imbalance ratio: 4.554545454545455
Imbalance ratio after SMOTE: 1.0


## Train KAN classifier

In [19]:
print("Training KAN Classifier...")
kan_classifier = KAN(width=[train_data_classif['train_input'].shape[1], 2, 2], grid=3, k=3, device=device, ckpt_path=out_path)
# kan_classifier.speed()

def train_acc():
    return torch.mean((torch.argmax(kan_classifier(train_data_classif['train_input']), dim=1) == train_data_classif['train_label']).type(torch.float32))

def test_acc():
    return torch.mean((torch.argmax(kan_classifier(train_data_classif['test_input']), dim=1) == train_data_classif['test_label']).type(torch.float32))

results = kan_classifier.fit(train_data_classif, opt="LBFGS", steps=10, metrics=(train_acc, test_acc),
                             loss_fn=torch.nn.CrossEntropyLoss(), )
print("Training completed."
      f"\nFinal Train Accuracy: {results['train_acc'][-1]:.4f}"
      f"\nFinal Test Accuracy: {results['test_acc'][-1]:.4f}")

Training KAN Classifier...
checkpoint directory created: /home/alecacciatore/ECML26/GNN4eGFR/features_importance_score/kan_scores
saving model version 0.0


description:   0%|                                                           | 0/10 [00:00<?, ?it/s]

| train_loss: 6.83e-01 | test_loss: 3.47e+01 | reg: 3.30e+02 | : 100%|█| 10/10 [00:07<00:00,  1.40it

saving model version 0.1
Training completed.
Final Train Accuracy: 0.8190
Final Test Accuracy: 0.8211


## Prune KAN classifier

In [20]:
# prune the network
kan_classifier.prune()

# test the pruned network
print("Testing pruned KAN Classifier...")
print("Train Accuracy after pruning:", train_acc().item())
print("Test Accuracy after pruning:", test_acc().item())

saving model version 0.2
Testing pruned KAN Classifier...
Train Accuracy after pruning: 0.8190125823020935
Test Accuracy after pruning: 0.8210583925247192


In [26]:
# get feature importance scores and save to CSV
feature_importances = kan_classifier.feature_score
feature_importances = feature_importances.cpu().detach().numpy()

importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': feature_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)
importance_df.to_csv(os.path.join(out_path, "kan_feature_importances.csv"), index=False)

In [27]:
# get 50 most important features
most_important_features = importance_df.head(50)
most_important_features.to_csv(os.path.join(out_path, "kan_top50_feature_importances.csv"), index=False)

# create new dataset with only the top 50 features
top_50_features = most_important_features['Feature'].values
X_top50 = X[top_50_features]
X_top50.to_csv(os.path.join(out_path, "XY_top50_features.csv"), index=False)

In [29]:
# re train KAN with only top 50 features 
print("Re-training KAN Classifier with top 50 features...")
train_data_classif_top50 = {
    'train_input': torch.tensor(X_train[top_50_features].values, dtype=torch.float32).to(device),
    'train_label': torch.tensor(y_classif_train.values, dtype=torch.long).to(device),
    'test_input': torch.tensor(X_test[top_50_features].values, dtype=torch.float32).to(device),
    'test_label': torch.tensor(y_classif_test.values, dtype=torch.long).to(device)
}
kan_classifier_top50 = KAN(width=[train_data_classif_top50['train_input'].shape[1], 2, 2], grid=3, k=3, device=device, ckpt_path=out_path)
# kan_classifier.speed()
def train_acc_top50():
    return torch.mean((torch.argmax(kan_classifier_top50(train_data_classif_top50['train_input']), dim=1) == train_data_classif_top50['train_label']).type(torch.float32))
def test_acc_top50():
    return torch.mean((torch.argmax(kan_classifier_top50(train_data_classif_top50['test_input']), dim=1) == train_data_classif_top50['test_label']).type(torch.float32))

results_top50 = kan_classifier_top50.fit(train_data_classif_top50, opt="LBFGS", steps=10, metrics=(train_acc_top50, test_acc_top50),
                             loss_fn=torch.nn.CrossEntropyLoss(), )
print("Re-training completed."
      f"\nFinal Train Accuracy with top 50 features: {results_top50['train_acc_top50'][-1]:.4f}"
      f"\nFinal Test Accuracy with top 50 features: {results_top50['test_acc_top50'][-1]:.4f}")

# # save the model
# model_path = os.path.join(out_path, "kan_classifier_top50.pth")



Re-training KAN Classifier with top 50 features...
checkpoint directory created: /home/alecacciatore/ECML26/GNN4eGFR/features_importance_score/kan_scores
saving model version 0.0


description:   0%|                                                           | 0/10 [00:00<?, ?it/s]

| train_loss: 6.91e-01 | test_loss: 6.93e-01 | reg: 4.36e+02 | : 100%|█| 10/10 [00:03<00:00,  2.89it

saving model version 0.1
Re-training completed.
Final Train Accuracy with top 50 features: 0.8200
Final Test Accuracy with top 50 features: 0.8189
